In [8]:
import pandas as pd
import duckdb as ddb

### Wattnet data

In [28]:
cf_wattnet = pd.read_csv("/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/entsoe_wattnet/footprints.csv")

In [29]:
cf_wattnet.head(10)

,timestamp,carbon_footprint
0,2024-12-14T00:00:00Z,216.16
1,2024-12-14T00:15:00Z,205.08
2,2024-12-14T00:30:00Z,202.53
3,2024-12-14T00:45:00Z,204.84
4,2024-12-14T01:00:00Z,204.54
5,2024-12-14T01:15:00Z,206.28
6,2024-12-14T01:30:00Z,205.69
7,2024-12-14T01:45:00Z,205.87
8,2024-12-14T02:00:00Z,203.58
9,2024-12-14T02:15:00Z,201.18


In [30]:
cf_wattnet["timestamp"] = pd.to_datetime(cf_wattnet["timestamp"], utc=True)
cf_wattnet.head()

,timestamp,carbon_footprint
0,2024-12-14 00:00:00+00:00,216.16
1,2024-12-14 00:15:00+00:00,205.08
2,2024-12-14 00:30:00+00:00,202.53
3,2024-12-14 00:45:00+00:00,204.84
4,2024-12-14 01:00:00+00:00,204.54


### Node Snapshot data

In [22]:
baseline = pd.read_parquet("/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/node_snapshot.parquet")
#baseline = baseline[["timestamp", "simulated_power"]]
baseline.head()

,timestamp,node_name,node_group,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power
0,2024-12-14 00:00:00+00:00,00366801,a6177608,0.23,404.17,128,1100.0,386653.664062,366137.002604,0,0.0,0.0,0.0,404.17,404.17
1,2024-12-14 00:00:00+00:00,0049db0c,a6177608,0.21,415.00,128,1100.0,386653.667969,289728.684245,0,0.0,0.0,0.0,415.00,415.00
2,2024-12-14 00:00:00+00:00,0064a367,a6177608,99.88,784.17,128,1100.0,386653.667969,1589.156901,0,0.0,0.0,0.0,784.17,784.17
3,2024-12-14 00:00:00+00:00,0065ef1b,a6177608,0.03,410.00,128,1100.0,386653.664062,366362.100911,0,0.0,0.0,0.0,410.00,410.00
4,2024-12-14 00:00:00+00:00,05c5ef00,a6177608,18.52,465.00,128,1100.0,386653.667969,1432.229167,0,0.0,0.0,0.0,465.00,465.00


In [23]:
baseline["timestamp"] = pd.to_datetime(baseline["timestamp"], utc=True)

In [24]:
baseline.head()

,timestamp,node_name,node_group,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power
0,2024-12-14 00:00:00+00:00,00366801,a6177608,0.23,404.17,128,1100.0,386653.664062,366137.002604,0,0.0,0.0,0.0,404.17,404.17
1,2024-12-14 00:00:00+00:00,0049db0c,a6177608,0.21,415.00,128,1100.0,386653.667969,289728.684245,0,0.0,0.0,0.0,415.00,415.00
2,2024-12-14 00:00:00+00:00,0064a367,a6177608,99.88,784.17,128,1100.0,386653.667969,1589.156901,0,0.0,0.0,0.0,784.17,784.17
3,2024-12-14 00:00:00+00:00,0065ef1b,a6177608,0.03,410.00,128,1100.0,386653.664062,366362.100911,0,0.0,0.0,0.0,410.00,410.00
4,2024-12-14 00:00:00+00:00,05c5ef00,a6177608,18.52,465.00,128,1100.0,386653.667969,1432.229167,0,0.0,0.0,0.0,465.00,465.00


In [25]:
baseline = baseline[["timestamp", "simulated_power"]]
baseline.head()

,timestamp,simulated_power
0,2024-12-14 00:00:00+00:00,404.17
1,2024-12-14 00:00:00+00:00,415.00
2,2024-12-14 00:00:00+00:00,784.17
3,2024-12-14 00:00:00+00:00,410.00
4,2024-12-14 00:00:00+00:00,465.00


In [26]:
baseline_agg = (
    baseline
    .groupby("timestamp", as_index=False)["simulated_power"]
    .sum()
    .rename(columns={"simulated_power": "total_simulated_power"})
)
baseline_agg.head()

,timestamp,total_simulated_power
0,2024-12-14 00:00:00+00:00,44234.11
1,2024-12-14 00:03:00+00:00,44431.64
2,2024-12-14 00:06:00+00:00,44355.31
3,2024-12-14 00:09:00+00:00,44428.48
4,2024-12-14 00:12:00+00:00,44295.16


### Merging

In [31]:
node_and_cf = cf_wattnet.merge(baseline_agg, on="timestamp", how="left")
node_and_cf

,timestamp,carbon_footprint,total_simulated_power
0,2024-12-14 00:00:00+00:00,216.16,44234.11
1,2024-12-14 00:15:00+00:00,205.08,44229.97
2,2024-12-14 00:30:00+00:00,202.53,44251.65
3,2024-12-14 00:45:00+00:00,204.84,44457.12
4,2024-12-14 01:00:00+00:00,204.54,44497.53
...,...,...,...
11611,2025-04-13 22:45:00+00:00,121.27,38032.51
11612,2025-04-13 23:00:00+00:00,122.67,38171.46
11613,2025-04-13 23:15:00+00:00,126.58,38142.33
11614,2025-04-13 23:30:00+00:00,125.27,38073.67


### Plot